# 🏭 Project 12.2 — Factory Floor Grid Planner
**DSA for Mechatronics · Week 12 — Dynamic Programming**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from functools import lru_cache
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A mobile robot navigates a factory floor grid from top-left to bottom-right,
moving only right or down. Some cells contain obstacles (blocked).

1. **Count unique paths** — how many distinct routes exist? (with and without obstacles)
2. **Minimum-cost path** — find the route with the lowest total cell cost
3. **Longest increasing path** — find the longest path where each step moves
   to a strictly higher value (any direction: up/down/left/right)


## Step 1 — Generate factory floor grids

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
ROWS   = random.randint(4, 6)
COLS   = random.randint(5, 8)
OBS_P  = random.uniform(0.10, 0.20)   # obstacle probability

# Grid with costs (for min-cost path)
COST_GRID = [[random.randint(1, 9) for _ in range(COLS)] for _ in range(ROWS)]

# Obstacle grid (0 = free, 1 = obstacle) — ensure start and end are free
OBS_GRID = [[1 if random.random() < OBS_P else 0 for _ in range(COLS)]
            for _ in range(ROWS)]
OBS_GRID[0][0] = 0
OBS_GRID[ROWS-1][COLS-1] = 0

# Value grid for longest increasing path
VAL_GRID = [[random.randint(1, 20) for _ in range(COLS)] for _ in range(ROWS)]

def print_grid(grid, label):
    print(f"  {label}:")
    for row in grid:
        print("    " + "  ".join(f"{v:2}" for v in row))

print(f"Factory floor grid: {ROWS}×{COLS}")
print()
print_grid(COST_GRID, "Cost grid (for min-cost path)")
print()
print_grid(OBS_GRID,  "Obstacle grid (0=free, 1=blocked)")
print()
print_grid(VAL_GRID,  "Value grid (for longest increasing path)")


## Step 2 — Count unique paths (with and without obstacles)

In [ ]:
def unique_paths_no_obs(rows, cols):
    """
    Count unique paths from (0,0) to (rows-1, cols-1), only right/down.

    dp[r][c] = number of ways to reach cell (r, c).
    Base: dp[0][c] = 1 for all c (only way: go all right)
          dp[r][0] = 1 for all r (only way: go all down)
    Recurrence: dp[r][c] = dp[r-1][c] + dp[r][c-1]
    O(rows * cols) time and space.
    """
    dp = [[1]*cols for _ in range(rows)]
    for r in range(1, rows):
        for c in range(1, cols):
            dp[r][c] = dp[r-1][c] + dp[r][c-1]
    return dp[rows-1][cols-1], dp

def unique_paths_with_obs(grid):
    """
    Same DP, but if a cell has an obstacle (grid[r][c]==1) → dp[r][c] = 0.
    Blocked cells contribute 0 paths, naturally cutting off those routes.
    """
    rows, cols = len(grid), len(grid[0])
    dp = [[0]*cols for _ in range(rows)]
    # First row: 1 until we hit an obstacle
    for c in range(cols):
        if grid[0][c] == 1: break
        dp[0][c] = 1
    # First col: 1 until we hit an obstacle
    for r in range(rows):
        if grid[r][0] == 1: break
        dp[r][0] = 1
    for r in range(1, rows):
        for c in range(1, cols):
            if grid[r][c] == 1:
                dp[r][c] = 0
            else:
                dp[r][c] = dp[r-1][c] + dp[r][c-1]
    return dp[rows-1][cols-1], dp

paths_no_obs, dp_no_obs = unique_paths_no_obs(ROWS, COLS)
paths_obs,    dp_obs    = unique_paths_with_obs(OBS_GRID)
n_obstacles             = sum(OBS_GRID[r][c] for r in range(ROWS) for c in range(COLS))

print(f"Unique paths ({ROWS}×{COLS} grid):")
print()
print(f"  WITHOUT obstacles:")
for row in dp_no_obs:
    print("    " + "  ".join(f"{v:4}" for v in row))
print(f"  Total paths: {paths_no_obs}")
print()
print(f"  WITH {n_obstacles} obstacle(s) (blocked cells = 0):")
for r in range(ROWS):
    row_str = ""
    for c in range(COLS):
        if OBS_GRID[r][c] == 1:
            row_str += "   X"
        else:
            row_str += f"{dp_obs[r][c]:4}"
    print(f"    {row_str}")
print(f"  Total paths (with obstacles): {paths_obs}")
print(f"  Paths eliminated by obstacles: {paths_no_obs - paths_obs}")


## Step 3 — Minimum cost path (bottom-right destination)

In [ ]:
def min_cost_path(grid):
    """
    Find minimum cost path from (0,0) to (rows-1, cols-1).
    Can only move right or down.

    dp[r][c] = minimum cost to reach (r, c).
    Base: dp[0][0] = grid[0][0]
          dp[0][c] = dp[0][c-1] + grid[0][c]  (only from left)
          dp[r][0] = dp[r-1][0] + grid[r][0]  (only from above)
    Recurrence: dp[r][c] = grid[r][c] + min(dp[r-1][c], dp[r][c-1])
    O(rows * cols) time and space.
    """
    rows, cols = len(grid), len(grid[0])
    dp = [[0]*cols for _ in range(rows)]
    dp[0][0] = grid[0][0]
    for c in range(1, cols):
        dp[0][c] = dp[0][c-1] + grid[0][c]
    for r in range(1, rows):
        dp[r][0] = dp[r-1][0] + grid[r][0]
    for r in range(1, rows):
        for c in range(1, cols):
            dp[r][c] = grid[r][c] + min(dp[r-1][c], dp[r][c-1])
    # Backtrack to reconstruct path
    path = []
    r, c = rows-1, cols-1
    while r > 0 or c > 0:
        path.append((r, c))
        if r == 0: c -= 1
        elif c == 0: r -= 1
        elif dp[r-1][c] < dp[r][c-1]: r -= 1
        else: c -= 1
    path.append((0, 0))
    path.reverse()
    return dp[rows-1][cols-1], dp, path

min_cost, dp_cost, opt_path = min_cost_path(COST_GRID)

print(f"Minimum cost path ({ROWS}×{COLS} grid):")
print()
print(f"  Cost grid:")
for r, row in enumerate(COST_GRID):
    print("    " + "  ".join(
        f"[{v}]" if (r,c) in opt_path else f" {v} "
        for c, v in enumerate(row)))
print(f"  (bracketed cells = optimal path)")
print()
print(f"  DP table (cumulative min costs):")
for row in dp_cost:
    print("    " + "  ".join(f"{v:3}" for v in row))
print()
print(f"  Optimal path   : {opt_path}")
print(f"  Path length    : {len(opt_path)} cells")
print(f"  Minimum cost   : {min_cost}")
path_cost_sum = sum(COST_GRID[r][c] for r, c in opt_path)
print(f"  Sum of path    : {path_cost_sum} == {min_cost}  "
      f"{'✅' if path_cost_sum == min_cost else '❌'}")


## Step 4 — Longest increasing path in grid (DFS + memoization)

In [ ]:
def longest_increasing_path(grid):
    """
    Find the longest path where each step moves to a strictly larger value.
    Can move in all 4 directions (up, down, left, right).

    DFS + memoization (top-down 2D DP):
      dp[r][c] = length of longest increasing path starting at (r, c).
    
    Key insight: no visited set needed — strictly increasing constraint
    means we can never revisit a cell (values must go up each step).
    
    Recurrence: dp[r][c] = 1 + max(dp[nr][nc])
                             for all neighbours (nr, nc) where grid[nr][nc] > grid[r][c]
    
    O(rows * cols) — each cell computed at most once.
    """
    rows, cols = len(grid), len(grid[0])
    memo = {}
    DIRS = [(0,1),(0,-1),(1,0),(-1,0)]

    def dfs(r, c):
        if (r, c) in memo:
            return memo[(r, c)]
        best = 1   # just this cell
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] > grid[r][c]:
                best = max(best, 1 + dfs(nr, nc))
        memo[(r, c)] = best
        return best

    global_best = 0
    best_start  = (0, 0)
    for r in range(rows):
        for c in range(cols):
            val = dfs(r, c)
            if val > global_best:
                global_best = val
                best_start  = (r, c)

    # Reconstruct the actual path
    path_lip = []
    r, c = best_start
    path_lip.append((r, c))
    while True:
        moved = False
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if (0 <= nr < rows and 0 <= nc < cols
                    and grid[nr][nc] > grid[r][c]
                    and memo.get((nr, nc), 0) == memo[(r, c)] - 1):
                path_lip.append((nr, nc)); r, c = nr, nc; moved = True; break
        if not moved: break

    return global_best, best_start, path_lip, memo

lip_len, lip_start, lip_path, lip_memo = longest_increasing_path(VAL_GRID)

print(f"Longest increasing path in grid ({ROWS}×{COLS}):")
print()
print(f"  Value grid (path cells marked with *):")
for r in range(ROWS):
    row_str = "    "
    for c in range(COLS):
        v = VAL_GRID[r][c]
        marker = "*" if (r, c) in lip_path else " "
        row_str += f"{marker}{v:2} "
    print(row_str)
print()
print(f"  DP table (longest path FROM each cell):")
for r in range(ROWS):
    print("    " + "  ".join(f"{lip_memo.get((r,c),0):2}" for c in range(COLS)))
print()
print(f"  Longest path length : {lip_len}")
print(f"  Starting cell       : {lip_start}  value={VAL_GRID[lip_start[0]][lip_start[1]]}")
print(f"  Path (row,col)      : {lip_path}")
print(f"  Path values         : {[VAL_GRID[r][c] for r,c in lip_path]}")
strictly_inc = all(VAL_GRID[lip_path[i+1][0]][lip_path[i+1][1]] >
                   VAL_GRID[lip_path[i][0]][lip_path[i][1]]
                   for i in range(len(lip_path)-1))
print(f"  Strictly increasing : {'✅ YES' if strictly_inc else '❌ NO'}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " FACTORY FLOOR GRID PLANNER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  UNIQUE PATHS" + " "*(W-14) + "║")
print(f"║  {'Grid size':<28}: {ROWS}×{COLS} = {ROWS*COLS} cells{'':<11}║")
print(f"║  {'Paths (no obstacles)':<28}: {paths_no_obs:<26}║")
print(f"║  {'Obstacles':<28}: {n_obstacles:<26}║")
print(f"║  {'Paths (with obstacles)':<28}: {paths_obs:<26}║")
print("╠" + "═"*W + "╣")
print("║  MIN-COST PATH" + " "*(W-15) + "║")
print(f"║  {'Minimum cost':<28}: {min_cost:<26}║")
print(f"║  {'Path length':<28}: {len(opt_path)} cells{'':<20}║")
print("╠" + "═"*W + "╣")
print("║  LONGEST INCREASING PATH" + " "*(W-25) + "║")
print(f"║  {'Path length':<28}: {lip_len:<26}║")
print(f"║  {'Starting cell':<28}: {str(lip_start):<26}║")
path_vals = [VAL_GRID[r][c] for r,c in lip_path]
print(f"║  {'Path values':<28}: {str(path_vals):<26}║")
print(f"║  {'Strictly increasing':<28}: {'YES ✅' if strictly_inc else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about DP in this context?

*Your answer here:*

---

**Q2 — Memoization vs Tabulation — which did you find clearer, and why?**
Describe the trade-offs between top-down (memoization) and bottom-up (tabulation) for one of the problems in this project.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
